# Reproduce ChakraTS-OS-10M on fev-bench

Reruns [fev-bench](https://huggingface.co/spaces/autogluon/fev-leaderboard) for [ChakraTS-OS-10M](https://huggingface.co/yhatlabs/ChakraTS-OS-10M) with the benchmark's own evaluation code, then compares
each result with the submitted one in this folder.

- `MODE = "quick"` (default): the first 3 tasks, a few minutes.
- `MODE = "full"`: all 100 tasks. Use a GPU: about 45 minutes on one NVIDIA H200.

The model runs on your own machine with [yhatmodels](https://github.com/yhatlabs/yhatmodels). Its first download asks you to accept the model license once with a free yhatlabs.com account; in Colab or on a server, set `YHAT_MODELS_TOKEN` (copy it from https://yhatlabs.com/models/license).

In [ ]:
MODE = "quick"  # "quick" (3 tasks) or "full" (all 100)
FEV_COMMIT = "43b946826813997dd79a4a1ec4cd992a920e1455"  # autogluon/fev with this model's wrapper

In [ ]:
import os, subprocess, sys, urllib.request
HERE = os.getcwd()
if not os.path.exists("fev"):
    subprocess.run(["git", "clone", "-q", "https://github.com/autogluon/fev"], check=True)
subprocess.run(["git", "-C", "fev", "checkout", "-q", FEV_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./fev", "-r", "fev/models/chakrats-os/requirements.txt"], check=True)
if not os.path.exists("chakrats-os-10m.csv"):  # in Colab: the submitted file from GitHub
    urllib.request.urlretrieve("https://raw.githubusercontent.com/yhatlabs/yhatlabs-cookbook/main/benchmarks/fev_bench/ChakraTS-OS-10M/chakrats-os-10m.csv", "chakrats-os-10m.csv")

In [ ]:
# The model's license is accepted once per machine: locally the first download opens your browser; in Colab or on a
# server, set YHAT_MODELS_TOKEN (from https://yhatlabs.com/models/license) before running the next cell.

In [ ]:
os.makedirs("rerun", exist_ok=True)
cmd = [sys.executable, "../fev/models/evaluate.py", "-m", "chakrats-os", "-b", "../fev/benchmarks/fev_bench/tasks.yaml", "-n", "chakrats-os-10m", "-d"]
subprocess.run(cmd + (["-t", "3"] if MODE == "quick" else []), cwd="rerun", check=True)

In [ ]:
import pandas as pd
submitted = pd.read_csv("chakrats-os-10m.csv").set_index("task_name")
rerun = pd.read_csv("rerun/chakrats-os-10m.csv").set_index("task_name")
table = pd.DataFrame({"submitted": submitted.loc[rerun.index, "test_error"], "rerun": rerun["test_error"]})
table["difference %"] = 100 * (table["rerun"] / table["submitted"] - 1)
print(table.round(4).to_string())
print(f"{len(table)} tasks, largest difference {table['difference %'].abs().max():.3f}%")